In [6]:
import pandas as pd

In [7]:
df=pd.read_csv('username.csv')
df["username"]

0               rose_glow01
1            by_shreyabarai
2             nagarkoti3036
3        rakhi_kharote_2988
4                  fitokids
                ...        
36594      littlechamp_anav
36595           glam.with_g
36596       __palakoberoi__
36597            berripuff_
36598      simple._.glide20
Name: username, Length: 36599, dtype: object

In [9]:
df["username"].to_csv("usernames.txt", index=False, header=False)

In [4]:
!pip install selenium webdriver-manager pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# instagram_selenium_scraper.py
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import json
import logging
import os
import re
from typing import Dict, List, Optional
from datetime import datetime

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class InstagramScraper:
    """Instagram profile scraper using Selenium"""
    
    def __init__(self, headless: bool = False):
        """
        Initialize the Chrome driver
        
        Args:
            headless: Run in headless mode (no browser window)
        """
        self.options = Options()
        
        if headless:
            self.options.add_argument('--headless')
        
        # Additional options to avoid detection
        self.options.add_argument('--no-sandbox')
        self.options.add_argument('--disable-dev-shm-usage')
        self.options.add_argument('--disable-gpu')
        self.options.add_argument('--window-size=1920,1080')
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)
        
        # User agent to appear like a real browser
        self.options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        
        # Initialize driver
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        self.wait = WebDriverWait(self.driver, 10)
        
    def get_user_data(self, username: str) -> Dict:
        """
        Scrape Instagram profile data for a given username
        
        Args:
            username: Instagram username (without @)
            
        Returns:
            Dictionary with user data
        """
        url = f"https://www.instagram.com/{username}/"
        logger.info(f"🌐 Scraping: {url}")
        
        try:
            # Navigate to profile
            self.driver.get(url)
            time.sleep(1)  # Wait for page load
            
            # Wait for page to load
            try:
                self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            except TimeoutException:
                logger.warning("Page load timeout, continuing anyway...")
            
            # Try to detect if profile exists — cover all Instagram "not found" variants
            page_text = self.driver.page_source
            not_found_markers = [
                "Sorry, this page isn't available",
                "The link you followed may be broken",
                "Page Not Found",
                "page isn't available",
            ]
            if any(marker in page_text for marker in not_found_markers):
                logger.warning(f"⚠️  Profile not available, skipping: {username}")
                return {
                    "username": username,
                    "error": "Profile not found",
                    "scraped_at": datetime.now().isoformat()
                }
            # Extract data using robust methods
            data = {
                "username": username,
                "full_name": self._get_username(),
                "followers": self._get_followers(),
                "following": self._get_following(),
                "bio": self._get_bio(),
                "portfolio_link": self._get_portfolio_link(),
                "scraped_at": datetime.now().isoformat()
            }
            
            logger.info(f"✅ Successfully scraped: {username}")
            return data
            
        except Exception as e:
            logger.error(f"❌ Error scraping {username}: {e}")
            return {
                "username": username, 
                "error": str(e),
                "scraped_at": datetime.now().isoformat()
            }
    
    def _get_username(self) -> Optional[str]:
        """Get username from profile"""
        selectors = [
            '//header//h2//span',
            '//header//section[2]//div//div//div//div//div//h2//span',
            '//*[@id="mount_0_0_ND"]//header//h2//span',
            'header h2 span',
            '//span[@class="x1lliihq x1plvlek xryxfnj x1n2onr6 xyejjpt x15dsfln x193iq5w xeuugli x1fj9vlw x13faqbe x1vvkbs x1s928wv xhkezso x1gmr53x x1cpjm7i x1fgarty x1943h6x x1i0vuye xvs91rp x1s688f x5n08af"]'
        ]
        
        for selector in selectors:
            try:
                if selector.startswith('//'):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    text = element.text.strip()
                    if text and len(text) > 1 and not text.isdigit():
                        # Make sure it's not follower/following count
                        if 'follower' not in text.lower() and 'following' not in text.lower():
                            return text
            except:
                continue
        return None
    
    def _get_followers(self) -> Optional[str]:
        """Get followers count from profile"""
        selectors = [
            '//header//ul//li[1]//span',
            '//header//section[3]//ul//li[1]//div//a//span',
            '//*[@id="mount_0_0_ND"]//header//ul//li[1]//span',
            'header ul li:first-child span',
            '//li[1]//span[contains(@class, "_ac2a")]'
        ]
        
        for selector in selectors:
            try:
                if selector.startswith('//'):
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    # Clean up - remove commas and convert to int
                    return text.replace(',', '')
            except:
                continue
        return None
    
    def _get_following(self) -> Optional[str]:
        """Get following count from profile"""
        selectors = [
            '//header//ul//li[2]//span',
            '//header//section[3]//ul//li[2]//div//a//span',
            '//*[@id="mount_0_0_ND"]//header//ul//li[2]//span',
            'header ul li:nth-child(2) span',
            '//li[2]//span[contains(@class, "_ac2a")]'
        ]
        
        for selector in selectors:
            try:
                if selector.startswith('//'):
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text.replace(',', '')
            except:
                continue
        return None
    
    def _get_bio(self) -> Optional[str]:
        """
        Get bio from profile - USING YOUR EXACT SELECTORS
        """
        # Wait a moment for bio to load
        time.sleep(2)
        
        # Method 1: Using the exact class you provided for bio header
        bio_selectors = [
            # Your exact class from the bio header
            'span[class*="x1lliihq x1plvlek xryxfnj x1n2onr6 xyejjpt x15dsfln x193iq5w xeuugli x1fj9vlw x13faqbe x1vvkbs x1s928wv xhkezso x1gmr53x x1cpjm7i x1fgarty x1943h6x x1i0vuye xvs91rp x1s688f x5n08af"]',
            
            # The bio class you provided
            'span._ap3a._aaco._aacu._aacx._aad7._aade',
            'span._ap3a',
            'span._aade',
            
            # Using partial class match
            'span[class*="_ap3a"]',
            'span[class*="_aaco"]',
            'span[class*="_aad7"]',
            
            # Your XPath based on the classes
            '//span[@class="_ap3a _aaco _aacu _aacx _aad7 _aade"]',
            '//span[contains(@class, "_ap3a") and contains(@class, "_aaco")]',
            '//span[contains(@class, "_aacu")]',
            
            # Generic selectors
            'header section:last-child span._ap3a',
            'header div[dir="auto"] span._ap3a',
            'header ._ap3a._aaco',
            
            # Original selectors that might work
            '//header//div[contains(@class, "_aa_c")]//span//div//span',
            '//header//div[contains(@class, "_ab_9")]//span',
            '//header//section[4]//div//span//div//span',
            
            # CSS Selectors
            'header div[class*="_aa_c"] span div span',
            'header div[class*="_ab_9"] span',
            'header section:last-child div span div span',
            'header ._aa_c span div span',
        ]
        
        for selector in bio_selectors:
            try:
                if selector.startswith('//'):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    text = element.text.strip()
                    if text and len(text) > 1:
                        # Make sure it's bio (not follower/following count)
                        if 'follower' not in text.lower() and 'following' not in text.lower():
                            if not text.replace(',', '').isdigit():
                                logger.info(f"✅ Found bio: {text[:100]}...")
                                return text
            except Exception as e:
                continue
        
        # Method 2: Search for bio using div with dir="auto"
        try:
            bio_elements = self.driver.find_elements(By.CSS_SELECTOR, 'div[dir="auto"] span[class*="_ap3a"]')
            for element in bio_elements:
                text = element.text.strip()
                if text and len(text) > 5:
                    if 'follower' not in text.lower() and 'following' not in text.lower():
                        if not text.replace(',', '').isdigit():
                            return text
        except:
            pass
        
        # Method 3: Get all text from header and find bio
        try:
            header = self.driver.find_element(By.TAG_NAME, 'header')
            all_text = header.text.split('\n')
            
            # Find lines that look like bio (not numbers, not just usernames)
            for line in all_text:
                line = line.strip()
                if line and len(line) > 5:
                    # Skip if it's just numbers (followers/following)
                    if not line.replace(',', '').isdigit():
                        # Skip if it's a username pattern
                        if not line.startswith('@'):
                            # Skip if it's just a name
                            if ' ' in line and len(line.split()) > 1:
                                return line
        except:
            pass
        
        return None
    
    def _get_portfolio_link(self) -> Optional[str]:
        """
        Get portfolio link from profile - USING YOUR EXACT SELECTORS
        """
        # Wait a moment for link to load
        time.sleep(2)
        
        # Method 1: Using your exact div class
        link_selectors = [
            # Your exact class from the portfolio link
            'div._ap3a._aaco._aacw._aacz._aada._aade',
            'div[class*="_ap3a _aaco _aacw _aacz _aada _aade"]',
            'div._ap3a._aaco',
            'div._aacw',
            
            # Using partial class match
            'div[class*="_ap3a"][class*="_aaco"]',
            'div[class*="_aacw"]',
            'div[class*="_aacz"]',
            
            # Your XPath based on the classes
            '//div[@class="_ap3a _aaco _aacw _aacz _aada _aade"]',
            '//div[contains(@class, "_ap3a") and contains(@class, "_aaco")]',
            
            # Find by dir="auto" attribute
            'div[dir="auto"]._ap3a._aaco',
            'div[dir="auto"][class*="_ap3a"]',
            
            # Existing selectors
            '//header//a[contains(@href, "http")]',
            '//header//div[contains(@class, "_ac2b")]//a',
            '//header//section[4]//div//div[3]//div//a',
            
            # CSS Selectors
            'header a[href*="http"]',
            'header div[class*="_ac2b"] a',
            'header section:last-child a',
            'header ._aa_c a',
        ]
        
        for selector in link_selectors:
            try:
                if selector.startswith('//'):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    # Try to get href first
                    href = element.get_attribute('href')
                    if href and ('http' in href or 'https' in href):
                        if 'instagram.com' not in href:
                            logger.info(f"✅ Found portfolio link: {href}")
                            return href
                    
                    # If no href, get text (like your example: nyk0.page.link/m5n923WDNi69Zx1V8)
                    text = element.text.strip()
                    if text and '.' in text and ' ' not in text:
                        # Check if it looks like a URL
                        if 'http' not in text and len(text) > 5:
                            # Add http if missing
                            if not text.startswith('http'):
                                full_url = f"https://{text}"
                                logger.info(f"✅ Found portfolio link: {full_url}")
                                return full_url
                        elif text.startswith('http'):
                            logger.info(f"✅ Found portfolio link: {text}")
                            return text
            except Exception as e:
                continue
        
        # Method 2: Search for text that looks like a URL
        try:
            all_elements = self.driver.find_elements(By.XPATH, '//*')
            for element in all_elements:
                text = element.text.strip()
                if text and '.' in text and ' ' not in text and len(text) > 5:
                    # Check if it looks like a URL or shortened link
                    if 'http' in text or 'bit.ly' in text or 'link' in text or '.page.link' in text:
                        if 'instagram.com' not in text:
                            if not text.startswith('http'):
                                return f"https://{text}"
                            return text
        except:
            pass
        
        return None
    
    def bulk_scrape_with_append(self, usernames: List[str], output_csv: str, delay: float = 3.0) -> List[Dict]:
        """
        Bulk scrape and append results instantly to CSV
        
        Args:
            usernames: List of Instagram usernames
            output_csv: Output CSV filename
            delay: Delay between requests in seconds
            
        Returns:
            List of user data dictionaries
        """
        results = []
        success_count = 0
        fail_count = 0
        
        logger.info(f"🚀 Starting bulk scrape for {len(usernames)} users...")
        logger.info(f"📁 Saving to: {output_csv}")
        
        # Initialize CSV file with headers if it doesn't exist
        if not os.path.exists(output_csv):
            empty_df = pd.DataFrame(columns=['username', 'full_name', 'followers', 'following', 'bio', 'portfolio_link', 'scraped_at', 'error'])
            empty_df.to_csv(output_csv, index=False, encoding='utf-8')
            logger.info(f"📄 Created new CSV file: {output_csv}")
        
        for i, username in enumerate(usernames, 1):
            logger.info(f"📊 {i}/{len(usernames)}: Scraping {username}")
            
            # Scrape user
            # Scrape user — never let one bad user kill the loop
            try:
                data = self.get_user_data(username)
            except Exception as e:
                logger.error(f"❌ Unexpected error for {username}: {e}")
                data = {
                    "username": username,
                    "error": str(e),
                    "scraped_at": datetime.now().isoformat()
                }
            results.append(data)
            
            # Immediately append to CSV
            self._append_to_csv(data, output_csv)
            
            if 'error' in data and data['error']:
                fail_count += 1
                logger.info(f"⚠️  {username}: FAILED - {data['error']}")
            else:
                success_count += 1
                logger.info(f"✅ {username}: SUCCESS")
                bio = data.get('bio') or 'N/A'
                portfolio = data.get('portfolio_link') or 'N/A'
                logger.info(f"   Bio: {bio[:50]}...")
                logger.info(f"   Portfolio: {portfolio}")
            
            # Rate limiting
            if i < len(usernames):
                time.sleep(delay)
            
            # Save progress backup every 5 users
            if i % 5 == 0:
                self._save_progress_backup(results, f"backup_progress_{i}.json")
                logger.info(f"💾 Progress backup saved at {i} users")
        
        logger.info(f"✅ Completed scraping {len(results)} users")
        logger.info(f"📊 Success: {success_count}, Failed: {fail_count}")
        
        return results
    
    def _append_to_csv(self, data: Dict, filename: str):
        """Append a single record to CSV instantly"""
        try:
            row = {
                'username': data.get('username', ''),
                'full_name': data.get('full_name', ''),
                'followers': data.get('followers', ''),
                'following': data.get('following', ''),
                'bio': data.get('bio', ''),
                'portfolio_link': data.get('portfolio_link', ''),
                'scraped_at': data.get('scraped_at', datetime.now().isoformat()),
                'error': data.get('error', '')
            }
            
            df_row = pd.DataFrame([row])
            df_row.to_csv(filename, mode='a', header=False, index=False, encoding='utf-8')
            
        except Exception as e:
            logger.error(f"❌ Error appending to CSV: {e}")
    
    def _save_progress_backup(self, data: List[Dict], filename: str):
        """Save progress backup"""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            logger.error(f"❌ Error saving backup: {e}")
    
    def close(self):
        """Close the Chrome driver"""
        self.driver.quit()


# ===================== MAIN EXECUTION =====================

def main():
    """Main execution function"""
    input_csv = "11march.csv"
    username_column = "Username"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_csv = f"instagram_scraped_{timestamp}.csv"
    
    print("\n" + "="*60)
    print("📱 INSTAGRAM BULK SCRAPER (FIXED WITH YOUR SELECTORS)")
    print("="*60)
    
    # Load usernames from CSV
    try:
        df = pd.read_csv(input_csv)
        usernames = df[username_column].dropna().astype(str).tolist()
        usernames = [u.strip() for u in usernames if u.strip()]
        print(f"📄 Loaded {len(usernames)} usernames from {input_csv}")
        print(f"📋 First 5: {', '.join(usernames[:5])}")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    if not usernames:
        print("❌ No usernames found!")
        return
    
    response = input("\nContinue? (y/n): ").strip().lower()
    if response != 'y':
        print("❌ Cancelled")
        return
    
    scraper = InstagramScraper(headless=False)
    
    try:
        data = scraper.bulk_scrape_with_append(usernames, output_csv, delay=1.0)
        
        print("\n" + "="*60)
        print("📊 SUMMARY")
        print("="*60)
        
        success = sum(1 for d in data if 'error' not in d or not d.get('error'))
        failed = len(data) - success
        
        print(f"✅ Success: {success}")
        print(f"❌ Failed: {failed}")
        print(f"📁 Output: {output_csv}")
        
        # Show sample of scraped data
        print("\n📋 Sample Output:")
        for user in data[:3]:
            if 'error' not in user or not user.get('error'):
                print(f"\n✅ {user.get('username')}")
                print(f"   Bio: {user.get('bio', 'N/A')[:100]}...")
                print(f"   Portfolio: {user.get('portfolio_link', 'N/A')}")
        
    except KeyboardInterrupt:
        print("\n⏹️ Interrupted - Data saved so far!")
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        scraper.close()


if __name__ == "__main__":
    main()


📱 INSTAGRAM BULK SCRAPER (FIXED WITH YOUR SELECTORS)
❌ Error loading CSV: [Errno 2] No such file or directory: '11march.csv'


### Version 2 Consisten With Monodb Model

In [ ]:
# instagram_selenium_scraper.py
import time
import re
import json
import logging
import os
from typing import Dict, List, Optional, Any
from datetime import datetime, timezone

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)


# ============================================================
# URL CLASSIFIER — routes URLs into the correct field
# ============================================================
def classify_url(url: Optional[str]) -> Optional[tuple]:
    """
    Returns (field_name, cleaned_url) or None.
    field_name ∈ {instagram, youtube, twitter, tiktok, linkedin, website, other_links}
    """
    if not url:
        return None

    u = url.strip()
    if not u:
        return None

    # Add scheme if missing
    if not u.startswith(("http://", "https://")):
        u = "https://" + u

    low = u.lower()

    if "instagram.com" in low:
        return ("instagram", u)
    if "youtube.com" in low or "youtu.be" in low:
        return ("youtube", u)
    if "twitter.com" in low or "x.com" in low:
        return ("twitter", u)
    if "tiktok.com" in low:
        return ("tiktok", u)
    if "linkedin.com" in low:
        return ("linkedin", u)

    # Common aggregators → website
    if any(d in low for d in (
        "linktr.ee", "beacons.ai", "bio.link", "linkin.bio",
        "carrd.co", "stan.store", "komi.io", "taplink.cc",
    )):
        return ("website", u)

    # Anything else → website if it looks like a real site
    # Otherwise → other_links (e.g., mailto:, telegram, custom short links)
    if any(d in low for d in ("t.me", "telegram", "discord.gg", "snapchat.com")):
        return ("other_links", u)

    # Default: treat as website
    return ("website", u)


# ============================================================
# BIO PARSER — extracts structured data from raw IG bio
# ============================================================
class BioParser:
    """Parses an Instagram bio into structured fields."""

    EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}")

    PHONE_RE = re.compile(
        r"(?:(?:\+|00)\d{1,3}[\s\-\.]?)?"
        r"(?:\(?\d{2,4}\)?[\s\-\.]?)?"
        r"\d{3,4}[\s\-\.]?\d{3,4}"
    )

    URL_RE = re.compile(
        r"(?:https?://)?(?:www\.)?"
        r"[a-zA-Z0-9\-]+\.[a-zA-Z]{2,}(?:/[^\s]*)?"
    )

    WHATSAPP_RE = re.compile(
        r"(?:wa\.me/|whatsapp\.com/send\?phone=|api\.whatsapp\.com/send\?phone=)"
        r"(\d{6,15})",
        re.IGNORECASE,
    )

    KNOWN_CATEGORIES = {
        "fashion", "lifestyle", "beauty", "makeup", "travel", "food",
        "fitness", "health", "tech", "technology", "gaming", "music",
        "art", "photography", "design", "business", "finance", "education",
        "comedy", "dance", "parenting", "pets", "sports", "automobile",
        "motivation", "spirituality", "books", "diy", "home", "decor",
        "skincare", "hair", "yoga", "cooking", "vlog", "vlogger",
        "influencer", "creator", "blogger", "model", "actor", "singer",
        "review", "reviews", "unboxing", "shopping", "kids", "family",
        "outdoors", "nature", "adventure", "culinary", "bakery",
        "ugc", "content", "digital", "marketing", "brand", "collab",
    }

    SKIP_WORDS = {
        "dm", "dms", "collab", "collabs", "collaboration", "collaborations",
        "business", "inquiries", "inquiry", "contact", "email", "mail",
        "for", "follow", "check", "link", "below", "bio", "click",
        "subscribe", "youtube", "instagram", "facebook", "twitter",
        "tiktok", "snapchat", "telegram", "more",
    }

    EMOJI_RE = re.compile(
        "["
        "\U0001F1E0-\U0001F1FF"
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\u2600-\u26FF"
        "\u2700-\u27BF"
        "\u2665\u2661\u2764\u2763"      # hearts
        "\u2022\u00b7\u2027"            # bullets
        "]+",
        flags=re.UNICODE,
    )

    @classmethod
    def clean_text(cls, s: str) -> str:
        if not s:
            return ""
        s = cls.EMOJI_RE.sub(" ", s)
        s = re.sub(r"\s+", " ", s)
        return s.strip()

    @classmethod
    def parse(cls, bio: str, username: str) -> Dict[str, Any]:
        out: Dict[str, Any] = {
            "display_name": None,
            "category": None,
            "tags": [],
            "email": None,
            "phone": None,
            "whatsapp": None,
            "website": None,
            "instagram": None,
            "youtube": None,
            "twitter": None,
            "tiktok": None,
            "linkedin": None,
            "other_links": [],
            "location_hints": [],
        }
        if not bio:
            return out

        # Strip trailing "more" that leaks through
        bio = re.sub(r"\s*more\s*$", "", bio, flags=re.IGNORECASE)
        raw_lines = [ln.strip() for ln in bio.split("\n") if ln.strip()]

        # ── 1) Email ────────────────────────────────────
        for line in raw_lines:
            m = cls.EMAIL_RE.search(line)
            if m:
                out["email"] = m.group(0).lower()
                break

        # ── 2) WhatsApp ─────────────────────────────────
        m = cls.WHATSAPP_RE.search(bio)
        if m:
            digits = m.group(1)
            out["whatsapp"] = f"https://wa.me/{digits}"
            if not out["phone"]:
                out["phone"] = f"+{digits}"

        # ── 3) Phone ────────────────────────────────────
        if not out["phone"]:
            for line in raw_lines:
                if out["email"] and out["email"] in line.lower():
                    continue
                if "http" in line.lower() or ".com" in line.lower():
                    continue
                for m in cls.PHONE_RE.finditer(line):
                    digits = re.sub(r"\D", "", m.group(0))
                    if 10 <= len(digits) <= 15:
                        cand = m.group(0)
                        out["phone"] = cand if cand.startswith("+") else f"+{digits}"
                        break
                if out["phone"]:
                    break

        # ── 4) URLs — classify each into the right field ─
        seen_urls = set()
        for line in raw_lines:
            if "@" in line and cls.EMAIL_RE.search(line):
                continue
            for m in cls.URL_RE.finditer(line):
                candidate = m.group(0).rstrip(".,;:!?)]")
                if "instagram.com" in candidate.lower():
                    continue  # skip IG's own links
                if candidate in seen_urls:
                    continue
                seen_urls.add(candidate)

                classified = classify_url(candidate)
                if not classified:
                    continue
                field, clean = classified

                if field == "other_links":
                    out["other_links"].append(clean)
                elif out[field] is None:
                    out[field] = clean

        # ── 5) Categories ───────────────────────────────
        candidates: List[str] = []
        for line in raw_lines:
            if "@" in line or "http" in line.lower() or ".com" in line.lower():
                continue
            if any(sep in line for sep in "|,•·/"):
                for part in re.split(r"[|,•·/]", line):
                    cleaned = cls.clean_text(part).lower().strip()
                    if cleaned and cleaned in cls.KNOWN_CATEGORIES:
                        candidates.append(cleaned)

        if not candidates:
            for w in re.findall(r"[a-zA-Z]{3,}", bio.lower()):
                if w in cls.KNOWN_CATEGORIES:
                    candidates.append(w)

        seen_cats = set()
        tags: List[str] = []
        for c in candidates:
            if c not in seen_cats:
                seen_cats.add(c)
                tags.append(c)

        if tags:
            out["tags"] = tags
            out["category"] = tags[0]

        # ── 6) Display name ─────────────────────────────
        for line in raw_lines:
            cleaned = cls.clean_text(line)
            if not cleaned or len(cleaned) < 3:
                continue
            if "@" in cleaned:
                continue
            if any(x in cleaned.lower() for x in ("http", ".com", "linktr", "www.")):
                continue
            if re.fullmatch(r"[\d\s\+\-\(\)]+", cleaned):
                continue
            if any(w in cleaned.lower().split() for w in cls.SKIP_WORDS):
                continue
            out["display_name"] = cleaned
            break

        return out


# ============================================================
# INSTAGRAM SCRAPER
# ============================================================
class InstagramScraper:
    """Instagram profile scraper using Selenium."""

    def __init__(self, headless: bool = False):
        self.options = Options()
        if headless:
            self.options.add_argument("--headless")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--disable-dev-shm-usage")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--window-size=1920,1080")
        self.options.add_argument("--disable-blink-features=AutomationControlled")
        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option("useAutomationExtension", False)
        self.options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.driver.execute_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )
        self.wait = WebDriverWait(self.driver, 10)

    # ── Scrape ────────────────────────────────────────────
    def get_user_data(self, username: str) -> Dict:
        url = f"https://www.instagram.com/{username}/"
        logger.info(f"🌐 Scraping: {url}")
        try:
            self.driver.get(url)

            # Smart wait instead of blind sleep
            try:
                WebDriverWait(self.driver, 2).until(
                    EC.presence_of_element_located((By.TAG_NAME, "header"))
                )
            except TimeoutException:
                logger.warning("Header not found within 5s, continuing...")

            page_text = self.driver.page_source

            # Login wall?
            if "Log in to Instagram" in page_text[:5000]:
                logger.warning(f"⚠️ Login wall detected on {username}")
                return {
                    "username": username,
                    "error": "Login wall — slow down",
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                }

            # Not found?
            not_found_markers = [
                "Sorry, this page isn't available",
                "The link you followed may be broken",
                "Page Not Found",
                "page isn't available",
            ]
            if any(m in page_text for m in not_found_markers):
                return {
                    "username": username,
                    "error": "Profile not found",
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                }

            # Scrape fields
            display_name_raw = self._get_username()
            followers_raw = self._get_followers()
            following_raw = self._get_following()
            bio = self._get_bio()
            portfolio_link = self._get_portfolio_link()

            # Empty render retry
            if not bio and not followers_raw:
                logger.warning(f"⚠️ Empty render for {username}, retrying once...")
                self.driver.refresh()
                time.sleep(2)
                display_name_raw = self._get_username()
                followers_raw = self._get_followers()
                following_raw = self._get_following()
                bio = self._get_bio()
                portfolio_link = self._get_portfolio_link()

            # Parse bio
            parsed = BioParser.parse(bio or "", username)

            # Classify portfolio link into the right field
            portfolio_field: Dict[str, Optional[str]] = {}
            other_from_portfolio: List[str] = []
            if portfolio_link:
                c = classify_url(portfolio_link)
                if c:
                    field, clean = c
                    if field == "other_links":
                        other_from_portfolio.append(clean)
                    else:
                        portfolio_field[field] = clean

            display_name = display_name_raw or parsed["display_name"]

            # Merge: bio-parsed field wins if it has a value; else use portfolio
            def pick(field: str) -> Optional[str]:
                return parsed.get(field) or portfolio_field.get(field)

            # Merge other_links (dedupe, preserve order)
            merged_others: List[str] = []
            for u in (parsed.get("other_links") or []) + other_from_portfolio:
                if u and u not in merged_others:
                    merged_others.append(u)

            data = {
                "username": username,
                "display_name": display_name,
                "profile_image_url": None,
                "verified": False,
                "bio": bio,
                "languages": [],
                "category": parsed["category"],
                "tags": parsed["tags"],
                "location": None,
                "email": parsed["email"],
                "phone": parsed["phone"],
                "whatsapp": parsed["whatsapp"],
                "instagram": f"https://instagram.com/{username}",
                "youtube": pick("youtube"),
                "twitter": pick("twitter"),
                "tiktok": pick("tiktok"),
                "linkedin": pick("linkedin"),
                "website": pick("website"),
                "other_links": merged_others,
                "followers": self._to_int(followers_raw),
                "following": self._to_int(following_raw),
                "source": "instagram_selenium",
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }

            logger.info(f"✅ Scraped: {username}")
            return data

        except Exception as e:
            logger.error(f"❌ Error scraping {username}: {e}")
            return {
                "username": username,
                "error": str(e),
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }

    # ── Helpers ───────────────────────────────────────────
    @staticmethod
    def _to_int(v) -> Optional[int]:
        if v is None:
            return None
        try:
            return int(str(v).replace(",", "").strip())
        except (ValueError, TypeError):
            return None

    def _get_username(self) -> Optional[str]:
        selectors = [
            '//header//h2//span',
            '//header//section[2]//div//div//div//div//div//h2//span',
            '//*[@id="mount_0_0_ND"]//header//h2//span',
            'header h2 span',
        ]
        for selector in selectors:
            try:
                if selector.startswith("//"):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    text = el.text.strip()
                    if text and len(text) > 1 and not text.isdigit():
                        if "follower" not in text.lower() and "following" not in text.lower():
                            return text
            except Exception:
                continue
        return None

    def _get_followers(self) -> Optional[str]:
        selectors = [
            '//header//ul//li[1]//span',
            '//header//section[3]//ul//li[1]//div//a//span',
            '//*[@id="mount_0_0_ND"]//header//ul//li[1]//span',
            'header ul li:first-child span',
        ]
        for selector in selectors:
            try:
                if selector.startswith("//"):
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text.replace(",", "")
            except Exception:
                continue
        return None

    def _get_following(self) -> Optional[str]:
        selectors = [
            '//header//ul//li[2]//span',
            '//header//section[3]//ul//li[2]//div//a//span',
            '//*[@id="mount_0_0_ND"]//header//ul//li[2]//span',
            'header ul li:nth-child(2) span',
        ]
        for selector in selectors:
            try:
                if selector.startswith("//"):
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                text = element.text.strip()
                if text:
                    return text.replace(",", "")
            except Exception:
                continue
        return None

    def _expand_bio(self) -> None:
        """Click '… more' so the full bio is visible."""
        try:
            more_btn = self.driver.find_element(
                By.XPATH,
                '//header//span[contains(text(), "more")] | '
                '//header//button[contains(., "more")] | '
                '//header//div[@role="button" and contains(., "more")]'
            )
            self.driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(0.3)
        except Exception:
            pass

    def _get_bio(self) -> Optional[str]:
        # Smart wait for bio container
        try:
            WebDriverWait(self.driver, 2).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, 'header span[class*="_ap3a"]')
                )
            )
        except TimeoutException:
            pass

        # Expand "… more" first
        self._expand_bio()

        bio_selectors = [
            'span._ap3a._aaco._aacu._aacx._aad7._aade',
            'span._ap3a',
            'span._aade',
            'span[class*="_ap3a"]',
            'span[class*="_aaco"]',
            'header section:last-child span._ap3a',
            'header div[dir="auto"] span._ap3a',
        ]
        for selector in bio_selectors:
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    text = el.text.strip()
                    if text and len(text) > 1:
                        if "follower" not in text.lower() and "following" not in text.lower():
                            if not text.replace(",", "").isdigit():
                                # Strip stray trailing "more"
                                text = re.sub(r"\s*more\s*$", "", text, flags=re.IGNORECASE)
                                return text
            except Exception:
                continue
        return None

    def _get_portfolio_link(self) -> Optional[str]:
        try:
            WebDriverWait(self.driver, 2).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, 'header a[href*="http"]')
                )
            )
        except TimeoutException:
            pass

        link_selectors = [
            'div._ap3a._aaco._aacw._aacz._aada._aade',
            'div[class*="_ap3a"][class*="_aaco"]',
            'div[class*="_aacw"]',
            'header a[href*="http"]',
            'header a',
        ]
        for selector in link_selectors:
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    href = el.get_attribute("href")
                    if href and "instagram.com" not in href and "http" in href:
                        return href
                    text = el.text.strip()
                    if text and "." in text and " " not in text and len(text) > 5:
                        if "instagram.com" not in text:
                            return text if text.startswith("http") else f"https://{text}"
            except Exception:
                continue
        return None

    # ── Bulk ──────────────────────────────────────────────
    def bulk_scrape_with_append(
        self, usernames: List[str], output_csv: str, delay: float = 1.0
    ) -> List[Dict]:
        results: List[Dict] = []
        success_count = 0
        fail_count = 0

        logger.info(f"🚀 Starting bulk scrape for {len(usernames)} users...")
        logger.info(f"📁 Saving to: {output_csv}")

        headers = [
            "username", "display_name", "followers", "following",
            "bio", "category", "tags",
            "email", "phone", "whatsapp",
            "instagram", "youtube", "twitter", "tiktok", "linkedin",
            "website", "other_links",
            "source", "scraped_at", "error",
        ]
        if not os.path.exists(output_csv):
            pd.DataFrame(columns=headers).to_csv(
                output_csv, index=False, encoding="utf-8"
            )
            logger.info(f"📄 Created new CSV file: {output_csv}")

        for i, username in enumerate(usernames, 1):
            logger.info(f"📊 {i}/{len(usernames)}: Scraping {username}")
            try:
                data = self.get_user_data(username)
            except Exception as e:
                logger.error(f"❌ Unexpected error for {username}: {e}")
                data = {
                    "username": username,
                    "error": str(e),
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                }

            results.append(data)
            self._append_to_csv(data, output_csv)

            if data.get("error"):
                fail_count += 1
                logger.info(f"⚠️  {username}: FAILED — {data['error']}")
            else:
                success_count += 1
                logger.info(f"✅ {username}: SUCCESS")
                logger.info(f"   Category: {data.get('category')}")
                logger.info(f"   Tags:     {data.get('tags')}")
                logger.info(f"   Email:    {data.get('email')}")
                logger.info(f"   Phone:    {data.get('phone')}")
                logger.info(f"   WhatsApp: {data.get('whatsapp')}")
                logger.info(f"   YouTube:  {data.get('youtube')}")
                logger.info(f"   Website:  {data.get('website')}")
                logger.info(f"   Other:    {data.get('other_links')}")

            if i < len(usernames):
                time.sleep(delay)

            if i % 5 == 0:
                self._save_progress_backup(results, f"backup_progress_{i}.json")
                logger.info(f"💾 Backup saved at {i} users")

        logger.info(f"✅ Completed {len(results)} users")
        logger.info(f"📊 Success: {success_count}, Failed: {fail_count}")
        return results

    def _append_to_csv(self, data: Dict, filename: str):
        try:
            other = data.get("other_links") or []
            row = {
                "username": data.get("username", ""),
                "display_name": data.get("display_name", ""),
                "followers": data.get("followers", ""),
                "following": data.get("following", ""),
                "bio": data.get("bio", ""),
                "category": data.get("category", ""),
                "tags": ",".join(data.get("tags") or []),
                "email": data.get("email", ""),
                "phone": data.get("phone", ""),
                "whatsapp": data.get("whatsapp", ""),
                "instagram": data.get("instagram", ""),
                "youtube": data.get("youtube", ""),
                "twitter": data.get("twitter", ""),
                "tiktok": data.get("tiktok", ""),
                "linkedin": data.get("linkedin", ""),
                "website": data.get("website", ""),
                "other_links": " | ".join(other),
                "source": data.get("source", ""),
                "scraped_at": data.get("scraped_at", ""),
                "error": data.get("error", ""),
            }
            pd.DataFrame([row]).to_csv(
                filename, mode="a", header=False, index=False, encoding="utf-8"
            )
        except Exception as e:
            logger.error(f"❌ Error appending to CSV: {e}")

    def _save_progress_backup(self, data: List[Dict], filename: str):
        try:
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False, default=str)
        except Exception as e:
            logger.error(f"❌ Error saving backup: {e}")

    def close(self):
        self.driver.quit()


# ============================================================
# MAIN
# ============================================================
def main():
    input_csv = "./usernames_data/username.csv"
    username_column = "username"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_csv = f"./output_data/instagram_scraped_{timestamp}.csv"

    print("\n" + "=" * 60)
    print("📱 INSTAGRAM BULK SCRAPER — EXPANDED BIO + URL CLASSIFICATION")
    print("=" * 60)

    try:
        df = pd.read_csv(input_csv)
        usernames = df[username_column].dropna().astype(str).tolist()
        usernames = [u.strip() for u in usernames if u.strip()]
        print(f"📄 Loaded {len(usernames)} usernames from {input_csv}")
        print(f"📋 First 5: {', '.join(usernames[:5])}")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return

    if not usernames:
        print("❌ No usernames found!")
        return

    if input("\nContinue? (y/n): ").strip().lower() != "y":
        print("❌ Cancelled")
        return

    scraper = InstagramScraper(headless=False)
    try:
        data = scraper.bulk_scrape_with_append(usernames, output_csv, delay=1.0)

        print("\n" + "=" * 60)
        print("📊 SUMMARY")
        print("=" * 60)
        success = sum(1 for d in data if not d.get("error"))
        failed = len(data) - success
        print(f"✅ Success: {success}")
        print(f"❌ Failed:  {failed}")
        print(f"📁 Output:  {output_csv}")

        print("\n📋 Sample:")
        for user in data[:3]:
            if not user.get("error"):
                print(f"\n✅ @{user.get('username')}")
                print(f"   Name:     {user.get('display_name')}")
                print(f"   Category: {user.get('category')}  Tags: {user.get('tags')}")
                print(f"   Email:    {user.get('email')}")
                print(f"   Phone:    {user.get('phone')}")
                print(f"   WhatsApp: {user.get('whatsapp')}")
                print(f"   YouTube:  {user.get('youtube')}")
                print(f"   Website:  {user.get('website')}")
                print(f"   Other:    {user.get('other_links')}")

    except KeyboardInterrupt:
        print("\n⏹️ Interrupted — data saved so far!")
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        scraper.close()


if __name__ == "__main__":
    main()


📱 INSTAGRAM BULK SCRAPER — EXPANDED BIO + URL CLASSIFICATION
📄 Loaded 36599 usernames from username.csv
📋 First 5: rose_glow01, by_shreyabarai, nagarkoti3036, rakhi_kharote_2988, fitokids


2026-10-03 12:14:43,479 - INFO - ====== WebDriver manager ======
2026-10-03 12:14:43,567 - INFO - Get LATEST chromedriver version for google-chrome
2026-10-03 12:14:43,606 - INFO - Get LATEST chromedriver version for google-chrome
2026-10-03 12:14:43,638 - INFO - Get LATEST chromedriver version for google-chrome
2026-10-03 12:14:43,836 - INFO - WebDriver version 148.0.7778.178 selected
2026-10-03 12:14:43,850 - INFO - Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/148.0.7778.178/linux64/chromedriver-linux64.zip
2026-10-03 12:14:43,852 - INFO - About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/148.0.7778.178/linux64/chromedriver-linux64.zip
2026-10-03 12:14:44,267 - INFO - Driver downloading response is 200
2026-10-03 12:14:47,276 - INFO - Get LATEST chromedriver version for google-chrome
2026-10-03 12:14:47,623 - INFO - Driver has been saved in cache [/home/itm/.wdm/drivers/chromedriver/linux64/148.0.7778.178]
202


⏹️ Interrupted — data saved so far!


### fatsre Version3 Improved Locationa nd Followers and Followings

In [ ]:
# instagram_selenium_scraper.py
import time
import re
import json
import logging
import os
from typing import Dict, List, Optional, Any
from datetime import datetime, timezone

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)


# ============================================================
# URL CLASSIFIER — routes URLs into the correct field
# ============================================================
def classify_url(url: Optional[str]) -> Optional[tuple]:
    """
    Returns (field_name, cleaned_url) or None.
    field_name ∈ {instagram, youtube, twitter, tiktok, linkedin, website, other_links}
    """
    if not url:
        return None

    u = url.strip()
    if not u:
        return None

    # Add scheme if missing
    if not u.startswith(("http://", "https://")):
        u = "https://" + u

    low = u.lower()

    if "instagram.com" in low:
        return ("instagram", u)
    if "youtube.com" in low or "youtu.be" in low:
        return ("youtube", u)
    if "twitter.com" in low or "x.com" in low:
        return ("twitter", u)
    if "tiktok.com" in low:
        return ("tiktok", u)
    if "linkedin.com" in low:
        return ("linkedin", u)

    # Common aggregators → website
    if any(d in low for d in (
        "linktr.ee", "beacons.ai", "bio.link", "linkin.bio",
        "carrd.co", "stan.store", "komi.io", "taplink.cc",
    )):
        return ("website", u)

    # Anything else → website if it looks like a real site
    # Otherwise → other_links (e.g., mailto:, telegram, custom short links)
    if any(d in low for d in ("t.me", "telegram", "discord.gg", "snapchat.com")):
        return ("other_links", u)

    # Default: treat as website
    return ("website", u)


# ============================================================
# BIO PARSER — extracts structured data from raw IG bio
# ============================================================
class BioParser:
    """Parses an Instagram bio into structured fields."""

    EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}")

    PHONE_RE = re.compile(
        r"(?:(?:\+|00)\d{1,3}[\s\-\.]?)?"
        r"(?:\(?\d{2,4}\)?[\s\-\.]?)?"
        r"\d{3,4}[\s\-\.]?\d{3,4}"
    )

    URL_RE = re.compile(
        r"(?:https?://)?(?:www\.)?"
        r"[a-zA-Z0-9\-]+\.[a-zA-Z]{2,}(?:/[^\s]*)?"
    )

    # Extract all locations marked with 📍
    LOCATION_RE = re.compile(r"📍\s*([^📍\\n]+)", re.UNICODE)

    WHATSAPP_RE = re.compile(
        r"(?:wa\.me/|whatsapp\.com/send\?phone=|api\.whatsapp\.com/send\?phone=)"
        r"(\d{6,15})",
        re.IGNORECASE,
    )

    KNOWN_CATEGORIES = {
        "fashion", "lifestyle", "beauty", "makeup", "travel", "food",
        "fitness", "health", "tech", "technology", "gaming", "music",
        "art", "photography", "design", "business", "finance", "education",
        "comedy", "dance", "parenting", "pets", "sports", "automobile",
        "motivation", "spirituality", "books", "diy", "home", "decor",
        "skincare", "hair", "yoga", "cooking", "vlog", "vlogger",
        "influencer", "creator", "blogger", "model", "actor", "singer",
        "review", "reviews", "unboxing", "shopping", "kids", "family",
        "outdoors", "nature", "adventure", "culinary", "bakery",
        "ugc", "content", "digital", "marketing", "brand", "collab",
    }

    SKIP_WORDS = {
        "dm", "dms", "collab", "collabs", "collaboration", "collaborations",
        "business", "inquiries", "inquiry", "contact", "email", "mail",
        "for", "follow", "check", "link", "below", "bio", "click",
        "subscribe", "youtube", "instagram", "facebook", "twitter",
        "tiktok", "snapchat", "telegram", "more",
    }

    EMOJI_RE = re.compile(
        "["
        "\U0001F1E0-\U0001F1FF"
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\u2600-\u26FF"
        "\u2700-\u27BF"
        "\u2665\u2661\u2764\u2763"      # hearts
        "\u2022\u00b7\u2027"            # bullets
        "]+",
        flags=re.UNICODE,
    )

    @classmethod
    def clean_text(cls, s: str) -> str:
        if not s:
            return ""
        s = cls.EMOJI_RE.sub(" ", s)
        s = re.sub(r"\s+", " ", s)
        return s.strip()

    @classmethod
    def parse(cls, bio: str, username: str) -> Dict[str, Any]:
        out: Dict[str, Any] = {
            "display_name": None,
            "category": None,
            "tags": [],
            "email": None,
            "phone": None,
            "whatsapp": None,
            "website": None,
            "instagram": None,
            "youtube": None,
            "twitter": None,
            "tiktok": None,
            "linkedin": None,
            "other_links": [],
            "location_hints": [],
        }
        if not bio:
            return out

        # Extract all 📍 locations from the bio
        locations = []
        for match in cls.LOCATION_RE.findall(bio):
            location = cls.clean_text(match)
            if location:
                locations.append(location)
        out["location_hints"] = list(dict.fromkeys(locations))

        # Strip trailing "more" that leaks through
        bio = re.sub(r"\s*more\s*$", "", bio, flags=re.IGNORECASE)
        raw_lines = [ln.strip() for ln in bio.split("\n") if ln.strip()]

        # ── 1) Email ────────────────────────────────────
        for line in raw_lines:
            m = cls.EMAIL_RE.search(line)
            if m:
                out["email"] = m.group(0).lower()
                break

        # ── 2) WhatsApp ─────────────────────────────────
        m = cls.WHATSAPP_RE.search(bio)
        if m:
            digits = m.group(1)
            out["whatsapp"] = f"https://wa.me/{digits}"
            if not out["phone"]:
                out["phone"] = f"+{digits}"

        # ── 3) Phone ────────────────────────────────────
        if not out["phone"]:
            for line in raw_lines:
                if out["email"] and out["email"] in line.lower():
                    continue
                if "http" in line.lower() or ".com" in line.lower():
                    continue
                for m in cls.PHONE_RE.finditer(line):
                    digits = re.sub(r"\D", "", m.group(0))
                    if 10 <= len(digits) <= 15:
                        cand = m.group(0)
                        out["phone"] = cand if cand.startswith("+") else f"+{digits}"
                        break
                if out["phone"]:
                    break

        # ── 4) URLs — classify each into the right field ─
        seen_urls = set()
        for line in raw_lines:
            if "@" in line and cls.EMAIL_RE.search(line):
                continue
            for m in cls.URL_RE.finditer(line):
                candidate = m.group(0).rstrip(".,;:!?)]")
                if "instagram.com" in candidate.lower():
                    continue  # skip IG's own links
                if candidate in seen_urls:
                    continue
                seen_urls.add(candidate)

                classified = classify_url(candidate)
                if not classified:
                    continue
                field, clean = classified

                if field == "other_links":
                    out["other_links"].append(clean)
                elif out[field] is None:
                    out[field] = clean

        # ── 5) Categories ───────────────────────────────
        candidates: List[str] = []
        for line in raw_lines:
            if "@" in line or "http" in line.lower() or ".com" in line.lower():
                continue
            if any(sep in line for sep in "|,•·/"):
                for part in re.split(r"[|,•·/]", line):
                    cleaned = cls.clean_text(part).lower().strip()
                    if cleaned and cleaned in cls.KNOWN_CATEGORIES:
                        candidates.append(cleaned)

        if not candidates:
            for w in re.findall(r"[a-zA-Z]{3,}", bio.lower()):
                if w in cls.KNOWN_CATEGORIES:
                    candidates.append(w)

        seen_cats = set()
        tags: List[str] = []
        for c in candidates:
            if c not in seen_cats:
                seen_cats.add(c)
                tags.append(c)

        if tags:
            out["tags"] = tags
            out["category"] = tags[0]

        # ── 6) Display name ─────────────────────────────
        for line in raw_lines:
            cleaned = cls.clean_text(line)
            if not cleaned or len(cleaned) < 3:
                continue
            if "@" in cleaned:
                continue
            if any(x in cleaned.lower() for x in ("http", ".com", "linktr", "www.")):
                continue
            if re.fullmatch(r"[\d\s\+\-\(\)]+", cleaned):
                continue
            if any(w in cleaned.lower().split() for w in cls.SKIP_WORDS):
                continue
            out["display_name"] = cleaned
            break

        return out


# ============================================================
# INSTAGRAM SCRAPER
# ============================================================
class InstagramScraper:
    """Instagram profile scraper using Selenium."""

    def __init__(self, headless: bool = False):
        self.options = Options()

        # Return from driver.get() immediately; we control rendering wait ourselves.
        self.options.page_load_strategy = "none"

        if headless:
            self.options.add_argument("--headless=new")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--disable-dev-shm-usage")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--window-size=1920,1080")
        self.options.add_argument("--disable-blink-features=AutomationControlled")
        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option("useAutomationExtension", False)
        self.options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)

        # Hard limit for navigation. The page can continue rendering in Chrome.
        self.driver.set_page_load_timeout(2)

        self.driver.execute_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )
        self.wait = WebDriverWait(self.driver, 2)
    def _get_profile_image_url(self) -> Optional[str]:
        """
        Grab the avatar URL from the profile page.
        Prefers <meta og:image> — the img element's classes rotate constantly.
        """
        # Method 1: og:image meta (most stable)
        try:
            el = self.driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]')
            url = el.get_attribute("content")
            if url and url.startswith("http"):
                return url
        except Exception:
            pass

        # Method 2: fallback — first header img
        try:
            img = self.driver.find_element(By.CSS_SELECTOR, 'header img')
            url = img.get_attribute("src")
            if url and url.startswith("http"):
                return url
        except Exception:
            pass

        return None
    # ── Scrape ────────────────────────────────────────────
    def get_user_data(self, username: str) -> Dict:
        url = f"https://www.instagram.com/{username}/"
        logger.info(f"🌐 Scraping: {url}")
        try:
            # Fast navigation: never wait for the whole page.
            try:
                self.driver.get(url)
            except TimeoutException:
                # Expected when the 2-second navigation limit is reached.
                pass

            # Wait only up to 2 seconds for the profile header.
            try:
                WebDriverWait(self.driver, 2).until(
                    EC.presence_of_element_located((By.TAG_NAME, "header"))
                )
            except TimeoutException:
                logger.warning("Header not rendered within 2s, continuing...")

            page_text = self.driver.page_source

            # Login wall?
            if "Log in to Instagram" in page_text[:5000]:
                logger.warning(f"⚠️ Login wall detected on {username}")
                return {
                    "username": username,
                    "error": "Login wall — slow down",
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                }

            # Not found?
            not_found_markers = [
                "Sorry, this page isn't available",
                "The link you followed may be broken",
                "Page Not Found",
                "page isn't available",
            ]
            if any(m in page_text for m in not_found_markers):
                return {
                    "username": username,
                    "error": "Profile not found",
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                }

            # Scrape fields
            display_name_raw = self._get_username()
            followers_raw = self._get_followers()
            following_raw = self._get_following()
            bio = self._get_bio()
            portfolio_link = self._get_portfolio_link()

            # No refresh retry here: keep each profile within the fast 2-second target.

            # Parse bio
            parsed = BioParser.parse(bio or "", username)

            # Classify portfolio link into the right field
            portfolio_field: Dict[str, Optional[str]] = {}
            other_from_portfolio: List[str] = []
            if portfolio_link:
                c = classify_url(portfolio_link)
                if c:
                    field, clean = c
                    if field == "other_links":
                        other_from_portfolio.append(clean)
                    else:
                        portfolio_field[field] = clean

            display_name = display_name_raw or parsed["display_name"]

            # Merge: bio-parsed field wins if it has a value; else use portfolio
            def pick(field: str) -> Optional[str]:
                return parsed.get(field) or portfolio_field.get(field)

            # Merge other_links (dedupe, preserve order)
            merged_others: List[str] = []
            for u in (parsed.get("other_links") or []) + other_from_portfolio:
                if u and u not in merged_others:
                    merged_others.append(u)

            data = {
                "username": username,
                "display_name": display_name,
                "profile_image_url": self._get_profile_image_url(),
                "verified": False,
                "bio": bio,
                "languages": [],
                "category": parsed["category"],
                "tags": parsed["tags"],
                "location": parsed["location_hints"],
                "email": parsed["email"],
                "phone": parsed["phone"],
                "whatsapp": parsed["whatsapp"],
                "instagram": f"https://instagram.com/{username}",
                "youtube": pick("youtube"),
                "twitter": pick("twitter"),
                "tiktok": pick("tiktok"),
                "linkedin": pick("linkedin"),
                "website": pick("website"),
                "other_links": merged_others,
                "followers": self._to_int(followers_raw),
                "following": self._to_int(following_raw),
                "source": "instagram_selenium",
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }

            logger.info(f"✅ Scraped: {username}")
            return data

        except Exception as e:
            logger.error(f"❌ Error scraping {username}: {e}")
            return {
                "username": username,
                "error": str(e),
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }

    # ── Helpers ───────────────────────────────────────────
    @staticmethod
    def _to_int(v) -> Optional[int]:
        if v is None:
            return None

        try:
            value = str(v).replace(",", "").strip().lower()

            if value.endswith("k"):
                return int(float(value[:-1]) * 1_000)
            if value.endswith("m"):
                return int(float(value[:-1]) * 1_000_000)
            if value.endswith("b"):
                return int(float(value[:-1]) * 1_000_000_000)

            return int(float(value))
        except (ValueError, TypeError):
            return None

    def _get_username(self) -> Optional[str]:
        selectors = [
            '//header//h2//span',
            '//header//section[2]//div//div//div//div//div//h2//span',
            '//*[@id="mount_0_0_ND"]//header//h2//span',
            'header h2 span',
        ]
        for selector in selectors:
            try:
                if selector.startswith("//"):
                    elements = self.driver.find_elements(By.XPATH, selector)
                else:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    text = el.text.strip()
                    if text and len(text) > 1 and not text.isdigit():
                        if "follower" not in text.lower() and "following" not in text.lower():
                            return text
            except Exception:
                continue
        return None

    def _get_followers(self) -> Optional[str]:
        try:
            header = self.driver.find_element(By.TAG_NAME, "header")
            text = header.text

            match = re.search(
                r"([\d,.]+\s*[kKmMbB]?)\s+followers?",
                text,
                re.IGNORECASE,
            )
            if match:
                return match.group(1).replace(" ", "")

        except Exception as e:
            logger.warning(f"Could not extract followers: {e}")

        return None

    def _get_following(self) -> Optional[str]:
        try:
            header = self.driver.find_element(By.TAG_NAME, "header")
            text = header.text

            match = re.search(
                r"([\d,.]+\s*[kKmMbB]?)\s+following",
                text,
                re.IGNORECASE,
            )
            if match:
                return match.group(1).replace(" ", "")

        except Exception as e:
            logger.warning(f"Could not extract following: {e}")

        return None

    def _expand_bio(self) -> None:
        """Click '… more' so the full bio is visible."""
        try:
            more_btn = self.driver.find_element(
                By.XPATH,
                '//header//span[contains(text(), "more")] | '
                '//header//button[contains(., "more")] | '
                '//header//div[@role="button" and contains(., "more")]'
            )
            self.driver.execute_script("arguments[0].click();", more_btn)
        except Exception:
            pass

    def _get_bio(self) -> Optional[str]:
        # Smart wait for bio container
        try:
            WebDriverWait(self.driver, 1).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "header")
                )
            )
        except TimeoutException:
            pass

        # Expand "… more" first
        self._expand_bio()

        bio_selectors = [
            'span._ap3a._aaco._aacu._aacx._aad7._aade',
            'span._ap3a',
            'span._aade',
            'span[class*="_ap3a"]',
            'span[class*="_aaco"]',
            'header section:last-child span._ap3a',
            'header div[dir="auto"] span._ap3a',
        ]
        for selector in bio_selectors:
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    text = el.text.strip()
                    if text and len(text) > 1:
                        if "follower" not in text.lower() and "following" not in text.lower():
                            if not text.replace(",", "").isdigit():
                                # Strip stray trailing "more"
                                text = re.sub(r"\s*more\s*$", "", text, flags=re.IGNORECASE)
                                return text
            except Exception:
                continue
        return None

    def _get_portfolio_link(self) -> Optional[str]:
        try:
            WebDriverWait(self.driver, 1).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, 'header a[href*="http"]')
                )
            )
        except TimeoutException:
            pass

        link_selectors = [
            'div._ap3a._aaco._aacw._aacz._aada._aade',
            'div[class*="_ap3a"][class*="_aaco"]',
            'div[class*="_aacw"]',
            'header a[href*="http"]',
            'header a',
        ]
        for selector in link_selectors:
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    href = el.get_attribute("href")
                    if href and "instagram.com" not in href and "http" in href:
                        return href
                    text = el.text.strip()
                    if text and "." in text and " " not in text and len(text) > 5:
                        if "instagram.com" not in text:
                            return text if text.startswith("http") else f"https://{text}"
            except Exception:
                continue
        return None

    # ── Bulk ──────────────────────────────────────────────
    def bulk_scrape_with_append(
        self, usernames: List[str], output_csv: str, delay: float = 0.0
    ) -> List[Dict]:
        results: List[Dict] = []
        success_count = 0
        fail_count = 0

        logger.info(f"🚀 Starting bulk scrape for {len(usernames)} users...")
        logger.info(f"📁 Saving to: {output_csv}")

        headers = [
            "username", "display_name", "followers", "following",
            "bio", "category", "tags", "location",
            "email", "phone", "whatsapp",
            "instagram", "youtube", "twitter", "tiktok", "linkedin",
            "website", "other_links",
            "source", "scraped_at", "error",
        ]
        if not os.path.exists(output_csv):
            pd.DataFrame(columns=headers).to_csv(
                output_csv, index=False, encoding="utf-8"
            )
            logger.info(f"📄 Created new CSV file: {output_csv}")

        for i, username in enumerate(usernames, 1):
            logger.info(f"📊 {i}/{len(usernames)}: Scraping {username}")
            try:
                data = self.get_user_data(username)
            except Exception as e:
                logger.error(f"❌ Unexpected error for {username}: {e}")
                data = {
                    "username": username,
                    "error": str(e),
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                }

            results.append(data)
            self._append_to_csv(data, output_csv)

            if data.get("error"):
                fail_count += 1
                logger.info(f"⚠️  {username}: FAILED — {data['error']}")
            else:
                success_count += 1
                logger.info(f"✅ {username}: SUCCESS")
                logger.info(f"   Category: {data.get('category')}")
                logger.info(f"   Tags:     {data.get('tags')}")
                logger.info(f"   Location: {data.get('location')}")
                logger.info(f"   Email:    {data.get('email')}")
                logger.info(f"   Phone:    {data.get('phone')}")
                logger.info(f"   WhatsApp: {data.get('whatsapp')}")
                logger.info(f"   YouTube:  {data.get('youtube')}")
                logger.info(f"   Website:  {data.get('website')}")
                logger.info(f"   Other:    {data.get('other_links')}")

            if i < len(usernames):
                time.sleep(delay)

            if i % 5 == 0:
                self._save_progress_backup(results, f"backup_progress_{i}.json")
                logger.info(f"💾 Backup saved at {i} users")

        logger.info(f"✅ Completed {len(results)} users")
        logger.info(f"📊 Success: {success_count}, Failed: {fail_count}")
        return results

    def _append_to_csv(self, data: Dict, filename: str):
        try:
            other = data.get("other_links") or []
            row = {
                "username": data.get("username", ""),
                "display_name": data.get("display_name", ""),
                "followers": data.get("followers", ""),
                "following": data.get("following", ""),
                "bio": data.get("bio", ""),
                "category": data.get("category", ""),
                "tags": ",".join(data.get("tags") or []),
                "location": " | ".join(data.get("location") or []),
                "email": data.get("email", ""),
                "phone": data.get("phone", ""),
                "whatsapp": data.get("whatsapp", ""),
                "instagram": data.get("instagram", ""),
                "youtube": data.get("youtube", ""),
                "twitter": data.get("twitter", ""),
                "tiktok": data.get("tiktok", ""),
                "linkedin": data.get("linkedin", ""),
                "website": data.get("website", ""),
                "other_links": " | ".join(other),
                "source": data.get("source", ""),
                "scraped_at": data.get("scraped_at", ""),
                "error": data.get("error", ""),
            }
            pd.DataFrame([row]).to_csv(
                filename, mode="a", header=False, index=False, encoding="utf-8"
            )
        except Exception as e:
            logger.error(f"❌ Error appending to CSV: {e}")

    def _save_progress_backup(self, data: List[Dict], filename: str):
        try:
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False, default=str)
        except Exception as e:
            logger.error(f"❌ Error saving backup: {e}")

    def close(self):
        self.driver.quit()


# ============================================================
# MAIN
# ============================================================
def main():
    input_csv = "./usernames_data/username.csv"
    username_column = "username"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_csv = f"./output_data/instagram_scraped_{timestamp}.csv"

    print("\n" + "=" * 60)
    print("📱 INSTAGRAM BULK SCRAPER — EXPANDED BIO + URL CLASSIFICATION")
    print("=" * 60)

    try:
        df = pd.read_csv(input_csv)
        usernames = df[username_column].dropna().astype(str).tolist()
        usernames = [u.strip() for u in usernames if u.strip()]
        print(f"📄 Loaded {len(usernames)} usernames from {input_csv}")
        print(f"📋 First 5: {', '.join(usernames[:5])}")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return

    if not usernames:
        print("❌ No usernames found!")
        return

    if input("\nContinue? (y/n): ").strip().lower() != "y":
        print("❌ Cancelled")
        return

    scraper = InstagramScraper(headless=False)
    try:
        data = scraper.bulk_scrape_with_append(usernames, output_csv, delay=0.0)

        print("\n" + "=" * 60)
        print("📊 SUMMARY")
        print("=" * 60)
        success = sum(1 for d in data if not d.get("error"))
        failed = len(data) - success
        print(f"✅ Success: {success}")
        print(f"❌ Failed:  {failed}")
        print(f"📁 Output:  {output_csv}")

        print("\n📋 Sample:")
        for user in data[:3]:
            if not user.get("error"):
                print(f"\n✅ @{user.get('username')}")
                print(f"   Name:     {user.get('display_name')}")
                print(f"   Category: {user.get('category')}  Tags: {user.get('tags')}")
                print(f"   Email:    {user.get('email')}")
                print(f"   Phone:    {user.get('phone')}")
                print(f"   WhatsApp: {user.get('whatsapp')}")
                print(f"   YouTube:  {user.get('youtube')}")
                print(f"   Website:  {user.get('website')}")
                print(f"   Other:    {user.get('other_links')}")

    except KeyboardInterrupt:
        print("\n⏹️ Interrupted — data saved so far!")
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        scraper.close()


if __name__ == "__main__":
    main()


📱 INSTAGRAM BULK SCRAPER — EXPANDED BIO + URL CLASSIFICATION
📄 Loaded 36599 usernames from ./usernames_data/username.csv
📋 First 5: rose_glow01, by_shreyabarai, nagarkoti3036, rakhi_kharote_2988, fitokids


2026-10-03 12:32:07,028 - INFO - ====== WebDriver manager ======
2026-10-03 12:32:07,119 - INFO - Get LATEST chromedriver version for google-chrome
2026-10-03 12:32:07,165 - INFO - Get LATEST chromedriver version for google-chrome
2026-10-03 12:32:07,258 - INFO - Driver [/home/itm/.wdm/drivers/chromedriver/linux64/148.0.7778.178/chromedriver-linux64/chromedriver] found in cache
2026-10-03 12:32:07,890 - INFO - 🚀 Starting bulk scrape for 36599 users...
2026-10-03 12:32:07,890 - INFO - 📁 Saving to: ./output_data/instagram_scraped_20261003_123205.csv
2026-10-03 12:32:07,894 - INFO - 📄 Created new CSV file: ./output_data/instagram_scraped_20261003_123205.csv
2026-10-03 12:32:07,895 - INFO - 📊 1/36599: Scraping rose_glow01
2026-10-03 12:32:07,896 - INFO - 🌐 Scraping: https://www.instagram.com/rose_glow01/
2026-10-03 12:32:10,621 - INFO - ✅ Scraped: rose_glow01
2026-10-03 12:32:10,626 - INFO - ✅ rose_glow01: SUCCESS
2026-10-03 12:32:10,627 - INFO -    Category: None
2026-10-03 12:32:10,627 -